In [1]:
import pandas as pd

In [63]:
# Load the raw data
df = pd.read_parquet('c:/Users/Ciro/C/_UGR_PDI/RELIEF-T1D/Datasets/2025-12-05/Preprocessed/t1diabetesgranada/files_for_crossvalidation/windows_with_folds_horizon_4.parquet')
df

,x0,x1,x2,x3,x4,x5,x6,x7,y,patient_id,fold_0,fold_1,fold_2,fold_3,fold_4
0,99,92,86,85,85,87,88,93,187,LIB193263,train,test,train,train,train
1,92,86,85,85,87,88,93,106,206,LIB193263,train,test,train,train,train
2,86,85,85,87,88,93,106,134,223,LIB193263,train,test,train,train,train
3,85,85,87,88,93,106,134,165,234,LIB193263,train,test,train,train,train
4,85,87,88,93,106,134,165,187,236,LIB193263,train,test,train,train,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19421401,173,187,208,206,191,170,155,148,169,LIB194166,train,test,train,val,train
19421402,187,208,206,191,170,155,148,145,206,LIB194166,train,test,train,val,train
19421403,208,206,191,170,155,148,145,141,215,LIB194166,train,test,train,val,train
19421404,206,191,170,155,148,145,141,143,204,LIB194166,train,test,train,val,train


In [64]:
df_info = pd.read_parquet('c:/Users/Ciro/C/_UGR_PDI/RELIEF-T1D/Datasets/2025-12-05/Preprocessed/t1diabetesgranada/Patient_info.parquet')
df_info

,Patient_ID,Sex,Age,Number_of_days_with_measures,Number_of_measurements
0,LIB193263,M,55,648,60097
1,LIB193264,F,45,326,26786
2,LIB193265,F,39,581,46575
3,LIB193266,M,22,500,55307
4,LIB193267,F,40,647,55500
...,...,...,...,...,...
638,LIB194162,M,22,361,33439
639,LIB194163,M,37,192,17754
640,LIB194164,F,29,632,58349
641,LIB194165,F,35,259,23862


In [65]:
# Drop all columns of data form x0 to x7 and y
columns_to_drop = [f'x{i}' for i in range(8)] + ['y']
df_ids = df.drop(columns=columns_to_drop)
df_ids

,patient_id,fold_0,fold_1,fold_2,fold_3,fold_4
0,LIB193263,train,test,train,train,train
1,LIB193263,train,test,train,train,train
2,LIB193263,train,test,train,train,train
3,LIB193263,train,test,train,train,train
4,LIB193263,train,test,train,train,train
...,...,...,...,...,...,...
19421401,LIB194166,train,test,train,val,train
19421402,LIB194166,train,test,train,val,train
19421403,LIB194166,train,test,train,val,train
19421404,LIB194166,train,test,train,val,train


In [66]:
# Leave just the rows wiht unique data
df_ids_unique = df_ids.drop_duplicates()
df_ids_unique.reset_index(inplace=True, drop=True)
df_ids_unique

,patient_id,fold_0,fold_1,fold_2,fold_3,fold_4
0,LIB193263,train,test,train,train,train
1,LIB193264,train,train,test,train,train
2,LIB193265,train,train,train,test,train
3,LIB193266,train,train,train,train,test
4,LIB193267,train,val,val,test,val
...,...,...,...,...,...,...
638,LIB194162,train,train,test,train,train
639,LIB194163,train,test,train,train,val
640,LIB194164,train,train,test,train,train
641,LIB194165,test,train,train,train,train


# Filter by sex

In [67]:
df_info['Sex'].value_counts()

Sex
F    334
M    309
Name: count, dtype: int64

In [68]:
# Get two dataset, one with the ids of the females and other with the males
female_ids = df_info[df_info['Sex'] == 'F']['Patient_ID'] # 
female_ids.reset_index(drop=True, inplace=True)

# Filter ids form df_ids_unique by female ids
df_ids_unique_female = df_ids_unique[df_ids_unique['patient_id'].isin(female_ids)]
df_ids_unique_female.reset_index(drop=True, inplace=True)

# Filter ids from df_ids_unique by male ids
df_ids_unique_male = df_ids_unique[~df_ids_unique['patient_id'].isin(female_ids)]
df_ids_unique_male.reset_index(drop=True, inplace=True)
df_ids_unique_male

,patient_id,fold_0,fold_1,fold_2,fold_3,fold_4
0,LIB193263,train,test,train,train,train
1,LIB193266,train,train,train,train,test
2,LIB193269,train,train,train,train,test
3,LIB193304,test,train,train,train,train
4,LIB193309,train,train,train,train,test
...,...,...,...,...,...,...
304,LIB194156,val,train,train,test,train
305,LIB194160,train,val,val,train,test
306,LIB194161,val,train,test,train,train
307,LIB194162,train,train,test,train,train


In [69]:
# Save as csv filter by sex datasets
df_ids_unique_female.to_csv('T1DiabetesGranada/sex/T1DiabetesGranada_female_PH4.csv', index=False)
df_ids_unique_male.to_csv('T1DiabetesGranada/sex/T1DiabetesGranada_male_PH4.csv', index=False)

# Filter by age
By this ranges: 'Age < 31', '31 <= Age < 46', '46 <= Age < 65', 'Age >= 66'

In [70]:
ids_young = df_info[df_info['Age'] < 31]['Patient_ID']
ids_young.reset_index(drop=True, inplace=True)
ids_young_adult = df_info[(df_info['Age'] >= 31) & (df_info['Age'] < 46)]['Patient_ID']
ids_young_adult.reset_index(drop=True, inplace=True)
ids_adult = df_info[(df_info['Age'] >= 46) & (df_info['Age'] < 65)]['Patient_ID']
ids_adult.reset_index(drop=True, inplace=True)
ids_elder = df_info[df_info['Age'] >= 65]['Patient_ID']
ids_elder.reset_index(drop=True, inplace=True)
# Filter ids form ids_unique by age groups
df_ids_unique_young = df_ids_unique[df_ids_unique['patient_id'].isin(ids_young)]
df_ids_unique_young.reset_index(drop=True, inplace=True)
df_ids_unique_young_adult = df_ids_unique[df_ids_unique['patient_id'].isin(ids_young_adult)]
df_ids_unique_young_adult.reset_index(drop=True, inplace=True)
df_ids_unique_adult = df_ids_unique[df_ids_unique['patient_id'].isin(ids_adult)]
df_ids_unique_adult.reset_index(drop=True, inplace=True)
df_ids_unique_elder = df_ids_unique[df_ids_unique['patient_id'].isin(ids_elder)]
df_ids_unique_elder.reset_index(drop=True, inplace=True)


In [71]:
# Save as csv filter by age datasets
df_ids_unique_young.to_csv('T1DiabetesGranada/age/T1DiabetesGranada_young_PH4.csv', index=False)
df_ids_unique_young_adult.to_csv('T1DiabetesGranada/age/T1DiabetesGranada_young_adult_PH4.csv', index=False)
df_ids_unique_adult.to_csv('T1DiabetesGranada/age/T1DiabetesGranada_adult_PH4.csv', index=False)
df_ids_unique_elder.to_csv('T1DiabetesGranada/age/T1DiabetesGranada_elder_PH4.csv', index=False)
